In [142]:
import re
import ast
from dotenv import load_dotenv
from anthropic import Anthropic
from statistics import mean

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [143]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=0.55, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

### Creating the eval dataset


In [144]:
import json
def generate_dataset():
    prompt = """
    Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
    that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
    each representing task that requires Python, JSON, or a Regex to complete.

    Example output:
    ```json
    [
        {
            "task": "Description of task",
            "format": "python"
            "solution_criteria":"Key criteria for evaluating the solution"
        },
        ...additional
    ]
    ```
    
    * "format" must be one of: "python", "json", "regex"
    * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
    * Focus on tasks that do not require writing much code

    Please generate 7 objects.
    """

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])

    return json.loads(text)

In [145]:
dataset = generate_dataset()
print(dataset)

[{'task': "Write a Python function that extracts the AWS region from an S3 bucket ARN (e.g., 'arn:aws:s3:::my-bucket'). Return the region or None if not applicable.", 'format': 'python', 'solution_criteria': "Function correctly identifies that S3 bucket ARNs don't contain region info and returns None, or extracts region from S3 access point ARNs that do contain region information."}, {'task': "Create a JSON object that represents an AWS IAM policy allowing read-only access to a specific S3 bucket named 'my-data-bucket'.", 'format': 'json', 'solution_criteria': "Valid JSON with proper IAM policy structure, correct actions (s3:GetObject, s3:ListBucket), resources pointing to 'my-data-bucket', and Effect set to 'Allow'."}, {'task': 'Write a regular expression that matches valid AWS EC2 instance IDs (format: i-followed by 8 or 17 hexadecimal characters).', 'format': 'regex', 'solution_criteria': "Regex correctly matches 'i-0a1b2c3d' and 'i-0a1b2c3d4e5f6g7h8i', but rejects 'i-abc', 'ec2-ins

In [146]:
with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

### Grader model 

In [147]:
def grade_by_model(test_case, solution):
    eval_prompt = f"""
      You're an expert code reviewer. Evaluate this AI-generated solution.
      Task: {test_case["task"]}
      Evaluation criteria: {test_case["solution_criteria"]}
      Solution: {solution}

        Scoring guide:
        - 9-10: Correct, handles edge cases, clean code
        - 7-8: Correct but missing edge cases or minor issues
        - 5-6: Partially correct or significant issues
        - 1-4: Wrong or non-functional

        Provide JSON with "strengths", "weaknesses", "reasoning", "score".
    """
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])

    return json.loads(eval_text)

### Code based grading

In [148]:
def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)


### Feed through Claude and the grader to check the quality of the prompt

In [149]:
def run_prompt(test_case):
    format_map = {"python": "```python", "json": "```json", "regex": "```"}
    prefix = format_map.get(test_case["format"], "```")

    prompt = f"""
        Here is an example of a high-quality solution:
   
        Task: Write a regex that matches valid AWS S3 bucket names
        Solution: ^(?!.*--)[a-z0-9][a-z0-9\\-]{{1,61}}[a-z0-9]$
   
        Now solve this task with the same quality:
        {test_case["task"]}

        Requirements: {test_case["solution_criteria"]}
        """
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, prefix)
    output = chat(messages, stop_sequences=["```"])
    
    return output.strip()

In [150]:
def run_test_case(test_case):

    output = run_prompt(test_case)
    
    model_grade = grade_by_model(test_case, output)
    print(model_grade)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2
    
    return {
        "ouput": output,
        "test_case": test_case,
        "score": score,
        "reasoning":reasoning
    }

In [151]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")
    
    return results

In [152]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

{'strengths': ["Correctly identifies that S3 bucket ARNs don't contain region information and returns None", 'Properly handles S3 access point ARNs by extracting the region field', 'Good documentation with clear docstring and examples', 'Type hints are present (Optional[str])', 'Input validation checks for string type and correct prefix', 'Regex pattern is well-structured and captures the region correctly', 'All provided examples in docstring work correctly'], 'weaknesses': ["Regex pattern assumes exactly 12-digit account ID (\\d{12}), but doesn't account for S3 on Outposts ARNs which have different formats (e.g., 'arn:aws:s3-outposts:region:account-id:outpost/outpost-id/accesspoint/name')", "Doesn't validate that the extracted region follows AWS region naming conventions (could reject valid regions or accept invalid ones)", 'The account ID validation (\\d{12}) is strict but undocumented - should clarify this is intentional', 'Missing handling for S3 Multi-Region Access Point ARNs (arn

In [153]:
print(json.dumps(results, indent=2))

[
  {
    "ouput": "import re\nfrom typing import Optional\n\ndef extract_region_from_s3_arn(arn: str) -> Optional[str]:\n    \"\"\"\n    Extracts the AWS region from an S3 ARN.\n    \n    S3 bucket ARNs (arn:aws:s3:::bucket-name) don't contain region info, returns None.\n    S3 access point ARNs (arn:aws:s3:region:account-id:accesspoint/name) do contain region.\n    \n    Args:\n        arn: The S3 ARN string\n        \n    Returns:\n        The region string, or None if not applicable (bucket ARN or invalid format)\n        \n    Examples:\n        >>> extract_region_from_s3_arn('arn:aws:s3:::my-bucket')\n        None\n        >>> extract_region_from_s3_arn('arn:aws:s3:us-east-1:123456789012:accesspoint/my-ap')\n        'us-east-1'\n        >>> extract_region_from_s3_arn('arn:aws:s3:eu-west-1:123456789012:accesspoint/test')\n        'eu-west-1'\n    \"\"\"\n    if not isinstance(arn, str) or not arn.startswith('arn:aws:s3:'):\n        return None\n    \n    # Pattern: arn:aws:s3:[reg